# `quality_mask()` ? m?scara de qualidade

Preparar uma m?scara de exclus?o a partir de **QA_PIXEL + QA_RADSAT**, na grade de uma imagem de refer?ncia, e combin?-la com nodata dessa imagem.

**`True` = pixel inv?lido; `False` = pixel dispon?vel para avalia??o.** A m?scara n?o garante acur?cia nem substitui os crit?rios de cada algoritmo.

Execute os **quatro blocos de c?digo**, de cima para baixo. Os t?picos ?Consulta? ao final explicam par?metros e alternativas; n?o s?o passos adicionais obrigat?rios.

## Antes de executar: separe estes arquivos

| Campo no passo 1 | Arquivo | Unidade / tipo | Requisito |
| --- | --- | --- | --- |
| `qa_pixel_path` | QA_PIXEL original | Bits inteiros | Mesma cena e grade da refer?ncia |
| `qa_radsat_path` | QA_RADSAT original | Bits inteiros | Obrigat?rio, mesmo se n?o houver satura??o |
| `reference_path` | Banda ou produto de entrada que receber? a m?scara | GeoTIFF de uma banda | CRS, transforma??o e dimens?es iguais ?s QA |

N?o calibre QA nem interpole seus bits. Para recortes, use a mesma janela em todos os arquivos. A fun??o verifica a grade, mas n?o comprova que os arquivos pertencem ? mesma cena.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione esse ambiente como kernel.

## Passo 1 ? Preencher caminhos e escolher a sa?da

Substitua todos os `CAMINHO_...` por arquivos existentes. No Windows, por exemplo, `Path(r"C:\dados\cena_QA_PIXEL.TIF")`.

`exclude_water=False` preserva a ?gua. Use `True` quando a etapa seguinte exigir uma sele??o terrestre, como no exemplo Wang. Uma nova execu??o no mesmo destino substitui o GeoTIFF anterior.

In [ ]:
from pathlib import Path
import numpy as np
import rasterio
from pylstemp import quality_mask

qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")
reference_path = Path("CAMINHO_DA_IMAGEM_DE_REFERENCIA.tif")
exclude_water = False
saida = Path("resultados/quality_mask")

entradas_obrigatorias = {
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
    "reference_path": reference_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Pr?ximo passo: verificar a grade e ler QA.")
print("Pasta de sa?da:", saida.resolve())

## Passo 2 ? Ler as QA e a imagem de refer?ncia

A fun??o exige as duas QA, confere a grade e inclui nodata das pr?prias QA. A leitura abaixo identifica separadamente nodata e valores n?o finitos da refer?ncia. N?o h? reproje??o ou reamostragem.

In [ ]:
mascara_qa = quality_mask(
    qa_pixel_path=qa_pixel_path,
    qa_radsat_path=qa_radsat_path,
    reference_path=reference_path,
    exclude_water=exclude_water,
)

with rasterio.open(reference_path) as src:
    if src.count != 1:
        raise ValueError("Escolha uma imagem de refer?ncia com uma ?nica banda.")
    perfil = src.profile.copy()
    referencia = src.read(1, masked=True).astype("float64")

imagem = referencia.filled(np.nan)
mascara_nodata = np.ma.getmaskarray(referencia) | ~np.isfinite(imagem)
print("Dimens?es:", mascara_qa.shape)
print("Pixels exclu?dos por QA:", int(mascara_qa.sum()))
print("Pixels nodata/n?o finitos da refer?ncia:", int(mascara_nodata.sum()))

## Passo 3 ? Combinar QA com nodata e aplicar a m?scara

Use OR (`|`), pois qualquer motivo de exclus?o deve invalidar o pixel. Os totais individuais podem se sobrepor; n?o devem ser somados.

A imagem mascarada abaixo mant?m os valores originais v?lidos e usa NaN nos demais. Nas fun??es num?ricas da biblioteca, passe a m?scara booleana como `mask=mascara`.

In [ ]:
mascara = mascara_qa | mascara_nodata
imagem_com_mascara = np.where(mascara, np.nan, imagem)

print("Pixels exclu?dos na m?scara combinada:", int(mascara.sum()))
print("Pixels dispon?veis:", int((~mascara).sum()), "de", mascara.size)
print("?gua exclu?da:", exclude_water)

## Passo 4 ? Salvar a m?scara combinada

O GeoTIFF usa **uint8: 0 = v?lido, 1 = inv?lido** e preserva a grade da refer?ncia. N?o h? valor nodata na m?scara: tanto 0 quanto 1 s?o dados v?lidos. Ao reler, converta com `.astype(bool)`.

Essa sa?da inclui nodata apenas da refer?ncia escolhida. Para outra opera??o, acrescente tamb?m nodata das demais entradas, mantendo a mesma grade.

In [ ]:
saida.mkdir(parents=True, exist_ok=True)
arquivo_saida = saida / "quality_mask.tif"
perfil.update(driver="GTiff", dtype="uint8", count=1, nodata=None, compress="deflate")

with rasterio.open(arquivo_saida, "w", **perfil) as dst:
    dst.write(mascara.astype("uint8"), 1)
    dst.update_tags(
        units="boolean_true_is_invalid",
        valid_value="0",
        invalid_value="1",
        exclude_water=str(exclude_water).lower(),
        reference_nodata_included="true",
    )
print("M?scara salva:", arquivo_saida.resolve())

## Consulta ? Chamada e par?metros

```python
mascara_qa = quality_mask(
    qa_pixel_path=qa_pixel_path,
    qa_radsat_path=qa_radsat_path,
    reference_path=reference_path,
    exclude_water=False,
)
```

| Par?metro | Condi??o | Padr?o | O que informar |
| --- | --- | --- | --- |
| `qa_pixel_path` | Obrigat?rio | Sem padr?o | Caminho do GeoTIFF QA_PIXEL |
| `qa_radsat_path` | Obrigat?rio | Sem padr?o | Caminho do GeoTIFF QA_RADSAT |
| `reference_path` | Obrigat?rio, por nome | Sem padr?o | Caminho do GeoTIFF que define a grade |
| `exclude_water` | Opcional | `False` | `True` acrescenta a exclus?o de ?gua, bit 7 de QA_PIXEL |

A fun??o n?o recebe `method=`: executa uma ?nica opera??o. N?o recebe matrizes QA nem uma imagem em lugar de um caminho.

## Consulta ? Crit?rios de exclus?o

- **QA_PIXEL:** bits 0?5 ? fill, nuvem dilatada, cirrus, nuvem, sombra e neve. ?gua (bit 7) ? opcional.
- **QA_RADSAT:** satura??o ?ptica B2/B4/B5 e oclus?o de terreno. Esses bits n?o codificam satura??o t?rmica.
- **Nodata das QA:** incorporado pela fun??o.
- **Nodata/n?o finitos da refer?ncia:** incorporado explicitamente no passo 3.

As flags diagn?sticas de PWV/LST descrevem condi??es dos estimadores e s?o uma etapa diferente. A fun??o n?o infere identidade de cena, n?o corrige a atmosfera e n?o valida a aplica??o dos m?todos a outros sensores.

## Consulta ? Reutilizar em um c?lculo

Depois de carregar BT10 e emissividade B10 na mesma grade, combine tamb?m o nodata dessas entradas e passe a m?scara ? fam?lia desejada. O trecho abaixo ? uma alternativa de consulta, n?o um quinto passo obrigat?rio:

```python
from pylstemp import single_window

mascara_lst = mascara | ~np.isfinite(bt10) | ~np.isfinite(e10)
lst = single_window(
    bt10, emissivity_10=e10,
    method="avdan-2016", mask=mascara_lst,
)
```

Para uma pasta de cena, `LandsatScene.quality_mask()` prepara a m?scara QA da cena. `calibrated()` a aplica e a retorna para reutiliza??o nas etapas seguintes.

[Fluxo completo de QA](../docs/quality-inputs.md) ? [Refer?ncia da API](../docs/api-reference.md) ? [?ndice dos exemplos](README.md).